# Handwritten to Data — 方案 A（检测 + 识别）

**在 Kaggle 上运行，无需本地下载数据、无需本地 GPU。**

## 使用步骤
1. 加入竞赛：https://www.kaggle.com/competitions/handwritten-to-data
2. **Code → New Notebook**，右侧 **Input** 添加竞赛数据集 `handwritten-to-data`
3. **Settings → Accelerator → GPU T4 x2**，**Internet → On**（首次下载预训练权重）
4. **Run All**（或先设 `QUICK_DEMO = True` 跑通流程，再改 `False` 正式训练）
5. 生成 `/kaggle/working/submission.csv` 后，在竞赛页 **Submit Predictions** 上传

## 流程
- **阶段 1**：用 train 标注框训练 YOLOv8 做区域检测 + 类型分类
- **阶段 2**：裁剪行图像，微调 TrOCR 做文本识别
- **推理**：对 test 检测 → 裁剪 → 识别 → 写出 `submission.csv`

In [ ]:
# ============ 配置 ============
QUICK_DEMO = True          # True=少量数据快速跑通；False=完整训练（耗时数小时）
DET_EPOCHS = 3 if QUICK_DEMO else 30
REC_EPOCHS = 2 if QUICK_DEMO else 8
MAX_TRAIN_PAGES = 80 if QUICK_DEMO else None   # None = 全部 1330 页
MAX_REC_SAMPLES = 2000 if QUICK_DEMO else None  # None = 全部裁剪样本
BATCH_DET = 8
BATCH_REC = 8
IMG_SIZE_DET = 640 if QUICK_DEMO else 960
CONF_DET = 0.15
IOU_DET = 0.45

import os, json, random, shutil, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
# ============ 路径（Kaggle 竞赛数据） ============
def find_competition_root():
    candidates = [
        Path("/kaggle/input/handwritten-to-data"),
        Path("/kaggle/input/rukopys"),
    ]
    for p in Path("/kaggle/input").glob("*") if Path("/kaggle/input").exists() else []:
        candidates.append(p)
    for c in candidates:
        if (c / "train" / "metadata.jsonl").exists():
            return c
    raise FileNotFoundError(
        "找不到竞赛数据。请在 Notebook 右侧 Input 添加 handwritten-to-data 数据集。"
    )

ROOT = find_competition_root()
TRAIN_IMG_DIR = ROOT / "train" / "images"
TEST_IMG_DIR = ROOT / "test" / "images"
TRAIN_META = ROOT / "train" / "metadata.jsonl"
TEST_META = ROOT / "test" / "metadata.jsonl"
SAMPLE_SUB = ROOT / "sample_submission.csv"

WORK = Path("/kaggle/working")
YOLO_DIR = WORK / "yolo_rukopys"
REC_DIR = WORK / "rec_crops"
DET_WEIGHTS = WORK / "det_best.pt"
REC_MODEL_DIR = WORK / "trocr_uk"

for d in [YOLO_DIR, REC_DIR, REC_MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Data root:", ROOT)

In [ ]:
# ============ 读取 metadata ============
REGION_TYPES = ["handwritten", "printed", "formula", "table", "annotation", "image", "graph"]
TYPE2ID = {t: i for i, t in enumerate(REGION_TYPES)}

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_META)
test_rows = load_jsonl(TEST_META)
if MAX_TRAIN_PAGES:
    train_rows = train_rows[:MAX_TRAIN_PAGES]

def image_id_from_row(row):
    return Path(row["file_name"]).name

print(f"Train pages: {len(train_rows)}, Test pages: {len(test_rows)}")

## 阶段 1：准备 YOLO 数据并训练检测器

In [ ]:
def yolo_label_line(bbox, cls_id, w, h):
    x1, y1, x2, y2 = bbox
    x1, x2 = max(0, x1), min(w, x2)
    y1, y2 = max(0, y1), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return None
    xc = ((x1 + x2) / 2) / w
    yc = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return f"{cls_id} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n"

def build_yolo_dataset(rows, split_name):
    img_out = YOLO_DIR / "images" / split_name
    lbl_out = YOLO_DIR / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for row in tqdm(rows, desc=f"YOLO {split_name}"):
        img_name = image_id_from_row(row)
        src = TRAIN_IMG_DIR / img_name
        if not src.exists():
            continue
        dst = img_out / img_name
        if not dst.exists():
            shutil.copy2(src, dst)
        w, h = row["image_width"], row["image_height"]
        lines = []
        for reg in row.get("regions") or []:
            t = reg.get("type", "handwritten")
            if t not in TYPE2ID:
                continue
            line = yolo_label_line(reg["bbox"], TYPE2ID[t], w, h)
            if line:
                lines.append(line)
        with open(lbl_out / (Path(img_name).stem + ".txt"), "w") as f:
            f.writelines(lines)

# 90/10 train-val split
idx = list(range(len(train_rows)))
random.shuffle(idx)
n_val = max(1, int(0.1 * len(idx)))
val_idx = set(idx[:n_val])
tr_rows = [train_rows[i] for i in idx[n_val:]]
va_rows = [train_rows[i] for i in idx[:n_val]]

for p in [YOLO_DIR / "images" / "train", YOLO_DIR / "labels" / "train",
          YOLO_DIR / "images" / "val", YOLO_DIR / "labels" / "val"]:
    if p.exists():
        shutil.rmtree(p)

build_yolo_dataset(tr_rows, "train")
build_yolo_dataset(va_rows, "val")

yaml_path = YOLO_DIR / "data.yaml"
yaml_path.write_text(
    f"path: {YOLO_DIR}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"nc: {len(REGION_TYPES)}\n"
    f"names: {REGION_TYPES}\n",
    encoding="utf-8",
)
print("YOLO dataset ready:", yaml_path)

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO

det_model = YOLO("yolov8n.pt")  # 小模型，Kaggle GPU 友好；可改 yolov8s.pt
det_results = det_model.train(
    data=str(yaml_path),
    epochs=DET_EPOCHS,
    imgsz=IMG_SIZE_DET,
    batch=BATCH_DET,
    device=0 if DEVICE == "cuda" else "cpu",
    project=str(WORK),
    name="yolo_train",
    exist_ok=True,
    patience=5,
    verbose=True,
)

best_pt = Path(det_results.save_dir) / "weights" / "best.pt"
shutil.copy2(best_pt, DET_WEIGHTS)
det_model = YOLO(str(DET_WEIGHTS))
print("Detector saved:", DET_WEIGHTS)

## 阶段 2：裁剪文本行并微调 TrOCR

In [ ]:
def crop_region(img, bbox, pad=4):
    x1, y1, x2, y2 = bbox
    w, h = img.size
    x1 = max(0, int(x1) - pad)
    y1 = max(0, int(y1) - pad)
    x2 = min(w, int(x2) + pad)
    y2 = min(h, int(y2) + pad)
    if x2 <= x1 or y2 <= y1:
        return None
    return img.crop((x1, y1, x2, y2))

rec_samples = []  # (crop_path, text, region_type)
crop_train = REC_DIR / "train"
crop_val = REC_DIR / "val"
crop_train.mkdir(parents=True, exist_ok=True)
crop_val.mkdir(parents=True, exist_ok=True)

val_ids = {image_id_from_row(r) for r in va_rows}
counter = 0

for row in tqdm(train_rows, desc="Build REC crops"):
    img_name = image_id_from_row(row)
    img_path = TRAIN_IMG_DIR / img_name
    if not img_path.exists():
        continue
    img = Image.open(img_path).convert("RGB")
    split_dir = crop_val if img_name in val_ids else crop_train
    for j, reg in enumerate(row.get("regions") or []):
        t = reg.get("type", "handwritten")
        if t in ("image", "graph"):
            continue
        text = (reg.get("text") or "").strip()
        if not text:
            continue
        crop = crop_region(img, reg["bbox"])
        if crop is None:
            continue
        if crop.width < 8 or crop.height < 8:
            continue
        fname = f"{Path(img_name).stem}_{j}.jpg"
        out_path = split_dir / fname
        crop.save(out_path, quality=90)
        rec_samples.append((str(out_path), text, t))
        counter += 1
        if MAX_REC_SAMPLES and counter >= MAX_REC_SAMPLES:
            break
    if MAX_REC_SAMPLES and counter >= MAX_REC_SAMPLES:
        break

rec_train_df = pd.DataFrame(rec_samples, columns=["path", "text", "type"])
rec_train_df = rec_train_df[rec_train_df["path"].str.contains("/train/")]
rec_val_df = pd.DataFrame(rec_samples, columns=["path", "text", "type"])
rec_val_df = rec_val_df[rec_val_df["path"].str.contains("/val/")]
print("REC train:", len(rec_train_df), "val:", len(rec_val_df))

In [ ]:
!pip install -q transformers evaluate jiwer

from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
ocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

# 扩展词表以覆盖乌克兰语字符
chars = set()
for txt in pd.concat([rec_train_df["text"], rec_val_df["text"]]):
    chars.update(list(txt))
extra = sorted(chars - set(processor.tokenizer.get_vocab().keys()))
if extra:
    processor.tokenizer.add_tokens(extra)
    ocr_model.decoder.resize_token_embeddings(len(processor.tokenizer))
    print(f"Added {len(extra)} tokens to tokenizer")

ocr_model.config.eos_token_id = processor.tokenizer.eos_token_id
ocr_model.config.pad_token_id = processor.tokenizer.pad_token_id
ocr_model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
ocr_model.config.max_length = 128

MAX_LEN = 128
TARGET_H = 64

class RecDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        w, h = img.size
        new_w = max(1, int(w * (TARGET_H / h)))
        img = img.resize((new_w, TARGET_H), Image.BICUBIC)
        pixel_values = processor(img, return_tensors="pt").pixel_values.squeeze(0)
        labels = processor.tokenizer(
            row["text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        labels[labels == processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

train_ds = RecDataset(rec_train_df)
val_ds = RecDataset(rec_val_df) if len(rec_val_df) else RecDataset(rec_train_df.head(50))

args = Seq2SeqTrainingArguments(
    output_dir=str(REC_MODEL_DIR),
    per_device_train_batch_size=BATCH_REC,
    per_device_eval_batch_size=BATCH_REC,
    num_train_epochs=REC_EPOCHS,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    learning_rate=4e-5,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    fp16=DEVICE == "cuda",
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=ocr_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
)

trainer.train()
trainer.save_model(str(REC_MODEL_DIR))
processor.save_pretrained(str(REC_MODEL_DIR))
ocr_model = VisionEncoderDecoderModel.from_pretrained(str(REC_MODEL_DIR)).to(DEVICE)
ocr_model.eval()
print("Recognizer saved:", REC_MODEL_DIR)

## 推理 + 生成 submission.csv

In [ ]:
ID2TYPE = {i: t for t, i in TYPE2ID.items()}

@torch.inference_mode()
def recognize_crop(pil_img):
    w, h = pil_img.size
    new_w = max(1, int(w * (TARGET_H / max(h, 1))))
    img = pil_img.resize((new_w, TARGET_H), Image.BICUBIC)
    pv = processor(img, return_tensors="pt").pixel_values.to(DEVICE)
    gen = ocr_model.generate(pv, max_length=MAX_LEN)
    return processor.batch_decode(gen, skip_special_tokens=True)[0].strip()

def detect_regions(pil_img):
    """YOLO 检测 → list of {bbox, type}"""
    res = det_model.predict(
        source=np.array(pil_img),
        conf=CONF_DET,
        iou=IOU_DET,
        verbose=False,
    )[0]
    regions = []
    if res.boxes is None or len(res.boxes) == 0:
        return regions
    for box in res.boxes:
        cls_id = int(box.cls.item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        regions.append({
            "bbox": [int(x1), int(y1), int(x2), int(y2)],
            "type": ID2TYPE.get(cls_id, "handwritten"),
        })
    return regions

def sort_regions_reading_order(regions):
    """与官方评测一致的阅读顺序（center_y // 15, center_x）"""
    def key(r):
        x1, y1, x2, y2 = r["bbox"]
        cy = (y1 + y2) / 2
        cx = (x1 + x2) / 2
        return (int(cy) // 15, cx)
    return sorted(regions, key=key)

def predict_page(img_path):
    img = Image.open(img_path).convert("RGB")
    dets = sort_regions_reading_order(detect_regions(img))
    out = []
    for reg in dets:
        t = reg["type"]
        if t in ("image", "graph"):
            out.append({"bbox": reg["bbox"], "type": t, "text": ""})
            continue
        crop = crop_region(img, reg["bbox"])
        if crop is None:
            continue
        text = recognize_crop(crop) if t != "image" else ""
        out.append({"bbox": reg["bbox"], "type": t, "text": text})
    return out

# 试跑一页
demo_img = TEST_IMG_DIR / image_id_from_row(test_rows[0])
if demo_img.exists():
    demo_regs = predict_page(demo_img)
    print("Demo:", demo_img.name, "regions:", len(demo_regs))
    if demo_regs:
        print(demo_regs[0])

In [ ]:
# 按 sample_submission 顺序生成提交文件
if SAMPLE_SUB.exists():
    sub_df = pd.read_csv(SAMPLE_SUB)
    test_images = sub_df["image"].tolist()
else:
    test_images = sorted(p.name for p in TEST_IMG_DIR.glob("*.jpg"))
    sub_df = pd.DataFrame({"image": test_images, "regions": "[]"})

pred_regions = []
for img_name in tqdm(test_images, desc="Predict test"):
    img_path = TEST_IMG_DIR / img_name
    if not img_path.exists():
        pred_regions.append("[]")
        continue
    regs = predict_page(img_path)
    pred_regions.append(json.dumps(regs, ensure_ascii=False))

sub_df["regions"] = pred_regions
out_path = WORK / "submission.csv"
sub_df.to_csv(out_path, index=False)
print("Saved:", out_path, "shape:", sub_df.shape)
print(sub_df.head(2))